In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd



In [3]:
%store -r x_train
x_train= x_train

%store -r y_train
y_train= y_train

%store -r x_test
x_test= x_test

%store -r y_test
y_test= y_test

%store -r data
data=data

%store -r numerical_cols
numerical_cols=numerical_cols

%store -r categorical_cols
categorical_cols= categorical_cols



%store -r preprocessor
preprocessor=preprocessor



In [43]:
# tfidf_vectorizer = TfidfVectorizer(stop_words='english')
# tfidf_matrix_content = tfidf_vectorizer.fit_transform(data['Brand '])
# cosine_similarities_content = cosine_similarity(tfidf_matrix_content,tfidf_matrix_content)

In [ ]:
# Prepare the full feature set
# x_content is the feature set including content-based features
X_content = preprocessor.fit_transform(data) # feature vector for al auto
X_content.shape

(27, 155)

#### **Cosinus similarity**
Cosine Similarity misst:

„Wie ähnlich ist Auto i zu Auto j basierend auf ihren Features?“


In [27]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(X_content, X_content)
#cosine_sim
#print(cosine_sim)

In [29]:
car_index = 10
cos_index=cosine_sim[car_index] # similarity scores for the car at index 10
cos_index

array([-1.03855168e-01, -7.38319181e-02, -1.94909349e-01,  1.16525243e-01,
        4.02884649e-01, -8.67234617e-02,  3.74831864e-02, -3.44243256e-02,
        1.24700472e-01,  2.42588516e-01,  1.00000000e+00,  1.67983884e-01,
       -1.19700011e-01,  3.81682866e-01,  5.36055129e-02,  3.95558121e-01,
        3.40245530e-04,  1.98938137e-01, -2.51309136e-02, -2.97443282e-02,
       -3.15254847e-02,  1.70554655e-01,  5.32634944e-02,  7.45005919e-02,
        4.56772569e-02,  3.22190578e-01,  2.81081114e-01])

#### **Content-Based**
**iloc** wählt Zeilen per Position
(Index des Autos, Ähnlichkeitswert)

In [44]:
# Function to recommend similar cars based on cosine similarity
def recommend_similar_cars(car_index, top_n=10):


    # Get the cosine similarity scores for the item
    similarity_scores = list(enumerate(cosine_sim[car_index]))

    # Sort similar items by similarity score in descending order
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Get the indices of the top N similar items (excluding the first one which is the item itself)
    similar_indices = [i for i, _ in similarity_scores[1:top_n+1]]

    return data.iloc[similar_indices]


In [34]:
recommend_similar_cars(5, top_n=5)

,Brand,Model,year of manufacture,Engine power (kilowatt),Engine power (Horsepower),Transmission type,Emission group,Cubic capacity,Number of gears,CO2 (g/km),...,space_saver_spare_wheel,has_headlight_washer,has_foldable_front_passenger_seat,has_warning_light,license_plate_holder,luggage_compartment_safety_net,has_height_adjustable_passenger_seat,has_foldable_seat_tables,has_advertising_foil,Sale price
18,Ford,Ford Focus,2020,110,150,automatik,AN,1995,8,140,...,0,0,0,0,0,0,0,0,0,4900
26,Ford,transit connect,2020,74,100,manuell,AN,1499,6,0,...,0,0,0,0,0,0,0,0,0,3200
0,Hyundai,Nexo Fuel Cell Sports,2022,120,163,automatik,AN,0,6,0,...,0,0,1,0,1,0,1,0,1,8300
2,Hyundai,Nexo Fuel Cell Sports,2024,120,163,automatik,AN,0,1,0,...,1,0,0,0,0,0,1,0,0,8600
1,Renault,Grand Scenic BLUE,2020,88,120,manuell,AN,1750,6,142,...,1,0,1,1,0,0,1,0,0,6600


In [ ]:
def content_based_recommendations(train_data, item_name, top_n=10):
    # Check if the item name exists in the training data
    if item_name not in train_data['Brand'].values:
        print(f"Item '{item_name}' not found in the training data.")
        return pd.DataFrame()

    # Create a TF-IDF vectorizer for item descriptions
    #tfidf_vectorizer = TfidfVectorizer(stop_words='english')

    # Apply TF-IDF vectorization to item descriptions
    #tfidf_matrix_content = tfidf_vectorizer.fit_transform(train_data['Tags'])

    # Calculate cosine similarity between items based on descriptions
    #cosine_similarities_content = cosine_similarity(tfidf_matrix_content, tfidf_matrix_content)

    # Find the index of the item
    item_index = train_data[train_data['Brand'] == item_name].index[0]

    # Get the cosine similarity scores for the item
    similar_items = list(enumerate(cosine_similarities_content[item_index]))

    # Sort similar items by similarity score in descending order
    similar_items = sorted(similar_items, key=lambda x: x[1], reverse=True)

    # Get the top N most similar items (excluding the item itself)
    top_similar_items = similar_items[1:top_n+1]

    # Get the indices of the top similar items
    recommended_item_indices = [x[0] for x in top_similar_items]

    # Get the details of the top similar items
    recommended_items_details = train_data.iloc[recommended_item_indices][['Brand', 'Model', 'year of manufacture', 'Engine power (kilowatt)', 'Rating']]

    return recommended_items_details

In [42]:
content_based_recommendations(data, 'Model', top_n=5)

KeyError: 'Brand'

#### **Collaborative Filtering**
Empfiehlt Items basierend auf dem Verhalten vieler Nutzer, nicht auf Item-Features.